# House — démonstration du pipeline (package `congress_core` + `house`)

Notebook **fin** : il *importe* le package et affiche un **aperçu des sorties** figées.
Aucune logique métier ici (elle vit dans `congress_core`), aucun re-run, aucun appel API.

## 1. Le cœur partagé s'importe

In [1]:
import sys
from pathlib import Path
REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'congress_core').is_dir():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
import congress_core
from congress_core import schema, identity, amounts, tickers, quiver, crosscheck, sector_enrich, reporting
print('congress_core', congress_core.__version__, '— modules cœur importés')
TABLES = REPO / '0 HOUSE' / 'toutes_annees' / 'data_v1' / 'tables'

congress_core 0.1.0 — modules cœur importés


## 2. Étapes du pipeline House

`house.digital` : index XML → manifest → `parse_ptr` → identité (`congress_core.identity`) → finalize (`schema.natural_key_hash` + dédup per-lot) → validation Quiver.

`house.ocr` : census A/B/C → Vision (cache versionné) → tickers/identité → fusion digital+OCR.

## 3. Aperçu de la table FINALE (digital + OCR fusionnés)

In [2]:
import pandas as pd
YEARS = [2020,2021,2022,2023,2024,2025,2026]
final = pd.concat([pd.read_csv(TABLES/str(y)/f'06_house_{y}_FINAL.csv', dtype=str) for y in YEARS], ignore_index=True)
dig = (final['provenance']=='house-pdf-electronic').sum(); ocr = (final['provenance']=='house-pdf-ocr').sum()
print(f'FINAL {len(final)} txns = digital {dig} + OCR {ocr} | {final["declarant_name"].nunique()} déposants | sans bioguide {final["bioguide_id"].isna().sum()}')
final[['declarant_name','transaction_date','ticker','operation_type','amount_range','provenance']].head(8)

FINAL 81985 txns = digital 32676 + OCR 49309 | 268 déposants | sans bioguide 0


,declarant_name,transaction_date,ticker,operation_type,amount_range,provenance
0,Austin Scott,2020-11-13,PLUG,Sale,"$1,001 - $15,000",house-pdf-electronic
1,Austin Scott,2020-12-23,XOM,Sale,"$1,001 - $15,000",house-pdf-electronic
2,Austin Scott,2020-11-16,T,Purchase,"$15,001",house-pdf-electronic
3,Austin Scott,2020-12-07,BLDP,Purchase,"$1,001 - $15,000",house-pdf-electronic
4,Austin Scott,2020-12-09,BLDP,Sale,"$1,001 - $15,000",house-pdf-electronic
5,Judy Chu,2020-12-23,ALL,Sale (Partial),"$15,001",house-pdf-electronic
6,Austin Scott,2020-10-09,BE,Sale,"$1,001 - $15,000",house-pdf-electronic
7,Austin Scott,2020-10-30,BE,Purchase,"$1,001 - $15,000",house-pdf-electronic


## 4. Validation — statut par déposant (`crosscheck`)

Constat : Quiver/Kadoa/Stock Watcher sont aveugles au papier → l'OCR est la **source unique** des déposants `ocr_unique`.

In [3]:
q = pd.read_csv(TABLES/'_quiver_house_cache.csv')
status = crosscheck.per_filer_status(final, q, chamber='house')
print(crosscheck.summary(status).to_string(index=False))
print()
status[status['status']=='ocr_unique'][['name','our_total','our_ocr','quiver','status']].head(8)

          status  deposants  transactions  dont_ocr
         digital         31          1152        10
      ocr_unique          8          1926      1918
quiver_validable        211         78907     47381



,name,our_total,our_ocr,quiver,status
5,David P. Roe,1686,1686,0,ocr_unique
58,Francis Rooney,189,189,0,ocr_unique
110,Ann Wagner,38,30,0,ocr_unique
186,Gus M. Bilirakis,5,5,0,ocr_unique
207,Joseph P. Kennedy,3,3,0,ocr_unique
208,Nicole Malliotakis,3,3,0,ocr_unique
229,Tom Cole,1,1,0,ocr_unique
249,Billy Long,1,1,0,ocr_unique


## 5. Reproductibilité

Le filet `tests/regression/` prouve que le cœur reproduit les colonnes figées à l'identique (natural_key_hash, amount_midpoint, infer_asset_type, bioguide, PROMPT_SHA). Voir `docs/REFONTE_HOUSE.md`.